![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/1.Clinical_Named_Entity_Recognition_Model.ipynb)

# Complete Spark NLP Healthcare NER Pipeline

This comprehensive notebook combines all steps for:
1. **Dataset Selection & Loading** - Load healthcare datasets
2. **NER Pipeline Execution** - Using pre-trained Spark NLP Healthcare models
3. **Entity Extraction & Merging** - Extract and merge entities with priority
4. **CoNLL File Generation** - Converting NER predictions to CoNLL format
5. **Custom Model Training** - Training a custom NER model from CoNLL data
6. **Resume Training** - Continuing training on additional data
7. **Model Evaluation** - Evaluate model performance with metrics

**✅ Modularized Code:** All Python modules are now in the `src` package and imported automatically. The code is clean, maintainable, and follows best practices.

**🚀 GPU Support:** This notebook is optimized for GPU acceleration. Make sure to:
1. Select GPU runtime: Runtime → Change runtime type → GPU (T4 or better)
2. GPU will be automatically detected and used for model inference and training

---

## Healthcare NLP for Data Scientists Course

If you are not familiar with the components in this notebook, you can check [Healthcare NLP for Data Scientists Udemy Course](https://www.udemy.com/course/healthcare-nlp-for-data-scientists/) and the [MOOC Notebooks](https://github.com/JohnSnowLabs/spark-nlp-workshop/tree/master/Spark_NLP_Udemy_MOOC/Healthcare_NLP) for each component.

## 🚀 GPU Performance Tips

**For Maximum Performance:**

1. **Enable GPU Runtime:**
   - Go to: Runtime → Change runtime type → Hardware accelerator → GPU (T4 or better)
   - Restart runtime after changing

2. **Expected Speed Improvements:**
   - **NER Inference:** 3-5x faster with GPU
   - **Model Training:** 5-10x faster with GPU
   - **Batch Processing:** Can handle 2x larger batches

3. **GPU Memory Management:**
   - Colab free tier: T4 GPU (16GB VRAM)
   - Colab Pro: Better GPUs available
   - If you get OOM errors, reduce batch_size

4. **Monitoring GPU Usage:**
   - Run `!nvidia-smi` to check GPU utilization
   - GPU should show high usage during model inference and training

**Note:** The notebook automatically detects and uses GPU if available. No manual configuration needed!


# Section 1: Setup & License Configuration

This section sets up the environment, installs required libraries, and configures Spark NLP Healthcare license.

## 1.1 Upload License File

Upload your Spark NLP Healthcare license file (`spark_jsl.json`). This file should contain:
- `SECRET`: Your license secret
- `JSL_VERSION`: Spark NLP JSL version
- `PUBLIC_VERSION`: Spark NLP public version

In [1]:
import json
import os
from google.colab import files

# if 'spark_jsl.json' not in os.listdir():
#     print("Please upload your spark_jsl.json license file:")
#     license_keys = files.upload()
#     os.rename(list(license_keys.keys())[0], 'spark_jsl.json')

with open('/kaggle/input/spark-jsl/spark_jsl.json') as f:
    license_keys = json.load(f)

# Define license key-value pairs as local variables
locals().update(license_keys)
os.environ.update(license_keys)

print("✅ License keys loaded")
print(f"JSL Version: {license_keys.get('JSL_VERSION', 'N/A')}")
print(f"Public Version: {license_keys.get('PUBLIC_VERSION', 'N/A')}")

✅ License keys loaded
JSL Version: 6.1.1
Public Version: 6.1.3


In [2]:
import os
import shutil

folder = '/kaggle/working'

# Tüm dosya ve klasörleri sil
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # dosya veya sembolik bağlantıyı sil
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # klasörü içindekilerle birlikte sil
    except Exception as e:
        print(f'Failed to delete {file_path}. Reason: {e}')

print("✅ /kaggle/working içindeki tüm dosyalar silindi.")


✅ /kaggle/working içindeki tüm dosyalar silindi.


## 1.2 Install Java and Required Libraries

First, install Java (required for Spark), then install PySpark, Spark NLP, Spark NLP Healthcare, and additional dependencies.

**Note:** In Colab, Java is usually pre-installed, but we check and install if needed.

In [3]:
# Install Java (required for Spark) - Check if already installed
import os
import subprocess

try:
    # Check if Java is already installed
    java_version = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT, text=True)
    print(f"✅ Java is already installed: {java_version.split(chr(10))[0]}")
    
    # Try to find JAVA_HOME
    if 'JAVA_HOME' not in os.environ:
        # Common Java paths in Colab
        java_paths = [
            "/usr/lib/jvm/java-11-openjdk-amd64",
            "/usr/lib/jvm/java-8-openjdk-amd64",
            "/usr/lib/jvm/default-java"
        ]
        for path in java_paths:
            if os.path.exists(path):
                os.environ["JAVA_HOME"] = path
                print(f"✅ Set JAVA_HOME to: {path}")
                break
except Exception as e:
    # Java not found, install it using shell command
    print(f"Java check failed: {e}")
    print("Installing Java 11...")
    os.system('apt-get update -qq > /dev/null 2>&1')
    os.system('apt-get -y install -qq openjdk-11-jdk > /dev/null 2>&1')
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    print("✅ Java 11 installation attempted")

# Verify Java installation
try:
    java_check = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT, text=True)
    print(f"✅ Java verification: {java_check.split(chr(10))[0]}")
except:
    print("⚠️ Warning: Java verification failed. You may need to restart the runtime.")

# Install PyTorch (required for GPU support in Spark NLP)
# Check if GPU is available and install appropriate PyTorch version
import subprocess
gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
has_gpu = gpu_check.returncode == 0

if has_gpu:
    print("🚀 GPU detected! Installing PyTorch with CUDA support...")
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
else:
    print("Installing PyTorch (CPU version)...")
    !pip install -q torch torchvision torchaudio

# Install PySpark and Spark NLP
!pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION

# Install Spark NLP Healthcare
!pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Install Spark NLP Display Library for visualization
!pip install -q spark-nlp-display

# Install additional dependencies
!pip install -q pandas numpy tqdm requests

print("✅ All libraries installed successfully!")
if has_gpu:
    print("✅ GPU-accelerated PyTorch installed")

✅ Java is already installed: openjdk version "11.0.27" 2025-04-15
✅ Set JAVA_HOME to: /usr/lib/jvm/java-11-openjdk-amd64
✅ Java verification: openjdk version "11.0.27" 2025-04-15
🚀 GPU detected! Installing PyTorch with CUDA support...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 73.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 45.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 86.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 10.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 30.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 13.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

## 1.3 Initialize Spark Session with GPU Support

Start Spark session with Spark NLP Healthcare license and optimal configuration for NER tasks.

**GPU Acceleration:**
- This notebook automatically detects and uses GPU if available
- For best performance, use Runtime → Change runtime type → GPU (T4 or better)
- GPU will significantly speed up model inference and training

**Note:** If you encounter Java gateway errors, try:
1. Restart the runtime (Runtime → Restart runtime)
2. Run all cells from the beginning
3. Ensure you have enough RAM allocated (use Runtime → Change runtime type → High-RAM if needed)

In [ ]:
# Import modüller from src package
import sys
import os
from pathlib import Path

# Add src directory to path (for Kaggle/Colab environments)
# Try multiple paths to find src directory
src_paths = [
    Path('/kaggle/working/src'),
    Path('/kaggle/working/src'),
    Path('../src'),
    Path('src'),
    Path('/content/src')
]

src_path = None
for path in src_paths:
    if path.exists():
        src_path = path
        break

if src_path:
    sys.path.insert(0, str(src_path.parent))
    print(f"✅ Found src directory at: {src_path}")
else:
    print("⚠️  src directory not found. Creating minimal structure...")
    # Create src directory and copy modules if needed
    import shutil
    src_path = Path('/kaggle/working/src')
    src_path.mkdir(parents=True, exist_ok=True)
    print(f"   Created: {src_path}")
    print("   Note: You may need to upload src files manually in Kaggle/Colab")

# Import all modules
try:
    from src import (
        DatasetLoader,
        NERPipeline,
        CoNLLConverter,
        ModelTrainer,
        extract_entities_from_ner_results,
        NERVisualizer
    )
    print("✅ All modules imported successfully!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("   Please ensure src directory contains all required modules:")
    print("   - dataset_loader.py")
    print("   - ner_pipeline.py")
    print("   - conll_converter.py")
    print("   - model_trainer.py")
    print("   - entity_extractor.py")
    print("   - ner_visualizer.py")
    print("   - __init__.py")


In [4]:
import sparknlp
import sparknlp_jsl
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Check Java installation
import subprocess
java_version = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT, text=True)
print(f"Java version check:\n{java_version.split(chr(10))[0]}\n")

# Check GPU availability
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_count = torch.cuda.device_count()
        print(f"🚀 GPU Detected: {gpu_name} ({gpu_count} device(s))")
        print(f"   GPU will be used for accelerated inference and training")
    else:
        print("⚠️  No GPU detected. Using CPU mode.")
        print("   For faster performance, enable GPU: Runtime → Change runtime type → GPU")
except ImportError:
    print("⚠️  PyTorch not available. GPU check skipped.")
    gpu_available = False

# Spark configuration optimized for Colab with GPU support
params = {
    "spark.driver.memory": "8G",  # Reduced from 16G for Colab
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize": "2000M",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer"
}

# Add GPU-specific configurations if GPU is available
if gpu_available:
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": "/content/cache_pretrained",
        "spark.jsl.settings.storage.cluster_tmp_dir": "/content/cache_pretrained",
        # Enable GPU acceleration for Spark NLP
        "spark.jsl.settings.annotator.gpu": "true"
    })
    print("✅ GPU acceleration enabled in Spark configuration")

# Start Spark session with error handling
try:
    print("Starting Spark session...")
    spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
    
    print(f"✅ Spark NLP Version: {sparknlp.version()}")
    print(f"✅ Spark NLP JSL Version: {sparknlp_jsl.version()}")
    print("✅ Spark session initialized successfully")
    
    # Test the session
    test_df = spark.createDataFrame([("test",)], ["text"])
    test_df.show()
    print("✅ Spark session is working correctly")
    
except Exception as e:
    print(f"❌ Error starting Spark session: {e}")
    print("\nTrying alternative configuration...")
    
    # Try with minimal configuration
    params_minimal = {
        "spark.driver.memory": "4G",
        "spark.kryoserializer.buffer.max": "1000M"
    }
    
    try:
        spark = sparknlp_jsl.start(license_keys['SECRET'], params=params_minimal)
        print("✅ Spark session started with minimal configuration")
    except Exception as e2:
        print(f"❌ Failed with minimal config: {e2}")
        raise

spark

Java version check:
openjdk version "11.0.27" 2025-04-15

🚀 GPU Detected: Tesla T4 (2 device(s))
   GPU will be used for accelerated inference and training
✅ GPU acceleration enabled in Spark configuration
Starting Spark session...
:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-19a60869-5251-4bbf-ab5b-ddbffe6389dd;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.1.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	found com.g

✅ Spark NLP Version: 6.1.3
✅ Spark NLP JSL Version: 6.1.1
✅ Spark session initialized successfully


+----+
|text|
+----+
|test|
+----+

✅ Spark session is working correctly


# Section 2: Dataset Selection & Loading

This section includes the DatasetLoader module and loads a healthcare dataset for NER processing.

## 2.1 DatasetLoader Module

This module handles loading of healthcare datasets for NER tasks. It supports downloading datasets from URLs and preparing them for processing.

In [5]:
"""
Dataset Loader Module
Handles loading of healthcare datasets for NER tasks
"""

import pandas as pd
import requests
from pathlib import Path
from typing import Optional, Tuple
import os


class DatasetLoader:
    """Load and prepare healthcare datasets for NER tasks"""

    def __init__(self, data_dir: str = "data/raw"):
        """
        Initialize DatasetLoader

        Args:
            data_dir: Directory to store raw data
        """
        self.data_dir = Path(data_dir)
        self.data_dir.mkdir(parents=True, exist_ok=True)

    def download_mtsamples_classifier(self) -> pd.DataFrame:
        """
        Download mtsamples_classifier dataset from Spark NLP workshop

        Returns:
            DataFrame with text data
        """
        url = "https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp-workshop/master/tutorials/Certification_Trainings/Healthcare/data/mtsamples_classifier.csv"
        file_path = self.data_dir / "mtsamples_classifier.csv"

        if not file_path.exists():
            print(f"Downloading mtsamples_classifier dataset from {url}...")
            response = requests.get(url, timeout=60)
            response.raise_for_status()

            with open(file_path, "wb") as f:
                f.write(response.content)
            print(f"Dataset saved to {file_path}")
        else:
            print(f"Dataset already exists at {file_path}")

        df = pd.read_csv(file_path)
        return df

    def download_oncology_notes(self) -> pd.DataFrame:
        """
        Download oncology notes dataset from Spark NLP workshop

        Returns:
            DataFrame with text data
        """
        # Oncology notes are typically in a directory, we'll need to handle multiple files
        base_url = "https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp-workshop/master/tutorials/Certification_Trainings/Healthcare/data/oncology_notes"

        # This is a placeholder - actual implementation would need to list and download files
        # For now, we'll return a message
        print(
            "Oncology notes dataset requires manual download or specific file listing"
        )
        print(f"Base URL: {base_url}")
        return pd.DataFrame()

    def load_local_dataset(self, file_path: str) -> pd.DataFrame:
        """
        Load dataset from local file

        Args:
            file_path: Path to local dataset file

        Returns:
            DataFrame with data
        """
        file_path = Path(file_path)
        if not file_path.exists():
            raise FileNotFoundError(f"Dataset file not found: {file_path}")

        if file_path.suffix == ".csv":
            return pd.read_csv(file_path)
        elif file_path.suffix == ".parquet":
            return pd.read_parquet(file_path)
        else:
            raise ValueError(f"Unsupported file format: {file_path.suffix}")

    def prepare_text_dataframe(
        self,
        df: pd.DataFrame,
        text_column: str = "text",
        id_column: Optional[str] = None,
        remove_duplicates: bool = True,
    ) -> pd.DataFrame:
        """
        Prepare dataframe for NER pipeline

        Args:
            df: Input dataframe
            text_column: Name of column containing text
            id_column: Name of column containing document IDs (optional)
            remove_duplicates: If True, remove duplicate texts (keeps first occurrence)

        Returns:
            Prepared dataframe with 'text_id' and 'text' columns
        """
        result_df = df.copy()
        # Remove duplicate texts if requested (before creating text_id to preserve original IDs)
        if remove_duplicates:
            initial_count = len(result_df)
            result_df = result_df.drop_duplicates(subset=[text_column], keep='first')
            removed_count = initial_count - len(result_df)
            if removed_count > 0:
                print(f"⚠️  Removed {removed_count} duplicate texts (kept first occurrence)")
                print(f"   Original: {initial_count} texts → After deduplication: {len(result_df)} texts")

        
        # Create text_id if not provided
        if id_column is None:
            result_df["text_id"] = range(len(result_df))
        else:
            result_df["text_id"] = result_df[id_column]

        # Ensure text column exists
        if text_column not in result_df.columns:
            raise ValueError(f"Text column '{text_column}' not found in dataframe")

        # Select and rename columns
        result_df = result_df[["text_id", text_column]].copy()
        result_df.columns = ["text_id", "text"]

        return result_df

    def get_sample_texts(self, df: pd.DataFrame, n_samples: int = 10) -> pd.DataFrame:
        """
        Get sample texts from dataset

        Args:
            df: Input dataframe
            n_samples: Number of samples to return

        Returns:
            DataFrame with sample texts
        """
        return df.head(n_samples)


## 2.2 Load Dataset

Load and prepare the healthcare dataset for NER processing. We use the mtsamples_classifier dataset which contains clinical notes.

In [6]:
# Initialize dataset loader
data_dir = "data/raw"
loader = DatasetLoader(data_dir=data_dir)

# Download mtsamples_classifier dataset
print("Downloading dataset...")
df = loader.download_mtsamples_classifier()
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Dataset saved to data/raw/mtsamples_classifier.csv
Dataset shape: (638, 2)
Columns: ['category', 'text']


,category,text
0,Gastroenterology,PROCEDURES PERFORMED: Colonoscopy. INDICATION...
1,Gastroenterology,OPERATION 1. Ivor-Lewis esophagogastrectomy. ...
2,Gastroenterology,PREOPERATIVE DIAGNOSES: 1. Gastroesophageal r...
3,Gastroenterology,PROCEDURE: Colonoscopy. PREOPERATIVE DIAGNOSE...
4,Gastroenterology,PREOPERATIVE DIAGNOSIS: Right colon tumor. PO...


In [7]:
# Prepare text dataframe for NER pipeline (with duplicate removal)
# Note: remove_duplicates defaults to True, so duplicates will be automatically removed
text_df = loader.prepare_text_dataframe(df, text_column="text", id_column=None)
print(f"\n✅ Text dataframe shape: {text_df.shape}")
print(f"Sample texts:")
text_df.head(10)

# Additional duplicate check and statistics
print(f"\n{'='*60}")
print("DUPLICATE CHECK REPORT")
print(f"{'='*60}")
total_texts = len(text_df)
unique_texts = text_df['text'].nunique()
duplicate_count = total_texts - unique_texts
duplicate_percentage = (duplicate_count / total_texts * 100) if total_texts > 0 else 0

print(f"Total texts: {total_texts}")
print(f"Unique texts: {unique_texts}")
print(f"Duplicates: {duplicate_count} ({duplicate_percentage:.2f}%)")

if duplicate_count > 0:
    print(f"\n⚠️  WARNING: {duplicate_count} duplicate texts found!")
    print("   Duplicates can affect:")
    print("   - Model training (overfitting, bias)")
    print("   - Model evaluation (inflated metrics)")
    print("   - Visualization (redundant displays)")
    print("\n   ✅ Duplicates have been removed (kept first occurrence)")
else:
    print("\n✅ No duplicates found - dataset is clean!")

# Show most common duplicate texts (if any)
if duplicate_count > 0:
    print(f"\n📊 Top 5 most duplicated texts:")
    duplicate_texts = text_df[text_df.duplicated(subset=['text'], keep=False)]['text'].value_counts().head(5)
    for idx, (text, count) in enumerate(duplicate_texts.items(), 1):
        text_preview = text[:100] + "..." if len(text) > 100 else text
        print(f"   {idx}. Count: {count+1} - Preview: {text_preview}")

⚠️  Removed 19 duplicate texts (kept first occurrence)
   Original: 638 texts → After deduplication: 619 texts

✅ Text dataframe shape: (619, 2)
Sample texts:

DUPLICATE CHECK REPORT
Total texts: 619
Unique texts: 618
Duplicates: 1 (0.16%)

⚠️  WARNING: 1 duplicate texts found!
   Duplicates can affect:
   - Model training (overfitting, bias)
   - Model evaluation (inflated metrics)
   - Visualization (redundant displays)

   ✅ Duplicates have been removed (kept first occurrence)

📊 Top 5 most duplicated texts:


# Section 3: NER Pipeline Execution

This section creates and executes a Spark NLP Healthcare NER pipeline with multiple pre-trained models.

## 3.1 NER Pipeline Module

This module creates a NER pipeline with three models:
- `ner_clinical`: Clinical entities (diseases, procedures, etc.)
- `ner_deid_generic_augmented`: PHI (Protected Health Information) entities
- `ner_posology`: Drug and dosage entities (Drug, Dosage prioritized)

**Priority:** Posology and DeID models take priority over clinical model for overlapping entities.

In [8]:
"""
NER Pipeline Module
Creates and executes Spark NLP Healthcare NER pipeline with multiple models
"""

import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import (
    MedicalNerModel,
    NerConverter
)
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from typing import Optional, Dict, List
import warnings
warnings.filterwarnings('ignore')


class NERPipeline:
    """NER Pipeline with multiple pre-trained models"""
    
    def __init__(self, spark: SparkSession, license_secret: Optional[str] = None):
        """
        Initialize NER Pipeline
        
        Args:
            spark: SparkSession instance
            license_secret: Spark NLP Healthcare license secret (if not already configured)
        """
        self.spark = spark
        self.license_secret = license_secret
        self.pipeline = None
        self.models = {}
        
    def create_pipeline(self, prioritize_posology_deid: bool = True):
        """
        Create NER pipeline with multiple models
        
        Args:
            prioritize_posology_deid: If True, posology and deid models take priority
        """
        # Document Assembler
        document_assembler = DocumentAssembler()\
            .setInputCol("text")\
            .setOutputCol("document")\
            .setCleanupMode("shrink")
        
        # Sentence Detector
        sentence_detector = SentenceDetector()\
            .setInputCols(["document"])\
            .setOutputCol("sentence")\
            .setExplodeSentences(True)
        
        # Tokenizer
        tokenizer = Tokenizer()\
            .setInputCols(["sentence"])\
            .setOutputCol("token")
        
        # Word Embeddings (required for MedicalNerModel)
        # Clinical embeddings are used for all NER models
        print("Loading clinical word embeddings...")
        word_embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models")\
            .setInputCols(["sentence", "token"])\
            .setOutputCol("embeddings")
        print("✅ Clinical embeddings loaded")
        
        # NER Models
        # Note: MedicalNerModel requires 3 inputs: document, token, and word_embeddings
        # 1. Clinical NER Model
        print("Loading ner_clinical model...")
        ner_clinical = MedicalNerModel.pretrained("ner_clinical", "en", "clinical/models")\
            .setInputCols(["document", "token", "embeddings"])\
            .setOutputCol("ner_clinical")
        
        # 2. DeID Generic Augmented Model
        print("Loading ner_deid_generic_augmented model...")
        ner_deid = MedicalNerModel.pretrained("ner_deid_generic_augmented", "en", "clinical/models")\
            .setInputCols(["document", "token", "embeddings"])\
            .setOutputCol("ner_deid")
        
        # 3. Posology Model (for Drug and Dosage)
        print("Loading ner_posology model...")
        ner_posology = MedicalNerModel.pretrained("ner_posology", "en", "clinical/models")\
            .setInputCols(["document", "token", "embeddings"])\
            .setOutputCol("ner_posology")
        
        # Store models
        self.models = {
            'clinical': ner_clinical,
            'deid': ner_deid,
            'posology': ner_posology
        }
        
        # Create pipeline stages
        # Note: word_embeddings must come before NER models
        stages = [
            document_assembler,
            sentence_detector,
            tokenizer,
            word_embeddings,  # Required for MedicalNerModel
            ner_clinical,
            ner_deid,
            ner_posology
        ]
        
        # If prioritizing, we need to merge results
        # For now, we'll run all models and merge in post-processing
        if prioritize_posology_deid:
            # Add NerConverter for each model
            ner_converter_clinical = NerConverter()\
                .setInputCols(["document", "token", "ner_clinical"])\
                .setOutputCol("chunk_clinical")
            
            ner_converter_deid = NerConverter()\
                .setInputCols(["document", "token", "ner_deid"])\
                .setOutputCol("chunk_deid")
            
            ner_converter_posology = NerConverter()\
                .setInputCols(["document", "token", "ner_posology"])\
                .setOutputCol("chunk_posology")
            
            stages.extend([
                ner_converter_clinical,
                ner_converter_deid,
                ner_converter_posology
            ])
        
        self.pipeline = Pipeline(stages=stages)
        return self.pipeline
    
    def fit_transform(self, data):
        """
        Fit and transform data through pipeline
        
        Args:
            data: Spark DataFrame with 'text' column
            
        Returns:
            Transformed DataFrame with NER results
        """
        if self.pipeline is None:
            raise ValueError("Pipeline not created. Call create_pipeline() first.")
        
        model = self.pipeline.fit(data)
        result = model.transform(data)
        return result
    
    def filter_posology_entities(self, ner_result, keep_entities: List[str] = ["Drug", "Dosage"]):
        """
        Filter posology entities to keep only Drug and Dosage
        
        Args:
            ner_result: NER result from posology model
            keep_entities: List of entity types to keep
            
        Returns:
            Filtered NER results
        """
        # This would be implemented based on the actual structure of NER results
        # For now, this is a placeholder
        return ner_result
    
    def merge_ner_results(self, result_df, prioritize_posology_deid: bool = True):
        """
        Merge results from multiple NER models with priority
        
        Priority order (if prioritize_posology_deid=True):
        1. Posology (Drug, Dosage)
        2. DeID (PHI entities)
        3. Clinical (other clinical entities)
        
        Args:
            result_df: DataFrame with NER results from all models
            prioritize_posology_deid: Whether to prioritize posology and deid
            
        Returns:
            DataFrame with merged NER results
        """
        # This is a complex operation that requires:
        # 1. Extracting entities from each model
        # 2. Resolving conflicts based on priority
        # 3. Creating a unified entity list
        
        # For now, return the original dataframe
        # Full implementation would merge chunks with priority logic
        return result_df
    
    def extract_entities(self, result_df) -> List[Dict]:
        """
        Extract entities from pipeline results
        
        Args:
            result_df: DataFrame with NER results
            
        Returns:
            List of entity dictionaries with text_id, begin, end, chunk, entity
        """
        entities = []
        
        # Extract entities from each model
        # This is a simplified version - actual implementation would need
        # to handle the Spark DataFrame structure properly
        
        return entities



## 3.2 Create and Run NER Pipeline

Create the NER pipeline and run it on the dataset. This will extract entities using all three models.

In [9]:
# Create NER pipeline
ner_pipeline = NERPipeline(spark, license_secret=license_keys['SECRET'])
pipeline = ner_pipeline.create_pipeline(prioritize_posology_deid=True)

print("✅ NER pipeline created")
print("Models in pipeline:")
print("  - ner_clinical")
print("  - ner_deid_generic_augmented")
print("  - ner_posology")

# Check GPU status for inference
try:
    import torch
    if torch.cuda.is_available():
        print(f"\n🚀 GPU available for inference: {torch.cuda.get_device_name(0)}")
        print("   Model inference will use GPU acceleration")
    else:
        print("\n⚠️  No GPU detected - using CPU for inference")
        print("   Enable GPU for faster inference: Runtime → Change runtime type → GPU")
except:
    print("\n⚠️  GPU check unavailable")

Loading clinical word embeddings...
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[ | ]

25/11/09 20:32:19 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/11/09 20:32:19 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[ \ ]Download done! Loading the resource.
[OK!]
✅ Clinical embeddings loaded
Loading ner_clinical model...
ner_clinical download started this may take some time.
Approximate size to download 13.9 MB
[ | ]

25/11/09 20:33:12 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/11/09 20:33:13 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/11/09 20:33:14 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


ner_clinical download started this may take some time.
Approximate size to download 13.9 MB
[ / ]Download done! Loading the resource.
[ \ ]

...

Download done! Loading the resource.
[OK!]
Loading ner_posology model...
ner_posology download started this may take some time.
Approximate size to download 13.8 MB
[ | ]

25/11/09 20:33:25 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/11/09 20:33:25 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


ner_posology download started this may take some time.
Approximate size to download 13.8 MB


25/11/09 20:33:25 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Download done! Loading the resource.
[OK!]
✅ NER pipeline created
Models in pipeline:
  - ner_clinical
  - ner_deid_generic_augmented
  - ner_posology

🚀 GPU available for inference: Tesla T4
   Model inference will use GPU acceleration


In [10]:
# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(text_df)
print(f"Spark DataFrame created with {spark_df.count()} rows")
spark_df.show(5, truncate=100)

Spark DataFrame created with 619 rows
+-------+----------------------------------------------------------------------------------------------------+
|text_id|                                                                                                text|
+-------+----------------------------------------------------------------------------------------------------+
|      0| PROCEDURES PERFORMED: Colonoscopy. INDICATIONS: Renewed symptoms likely consistent with active f...|
|      1| OPERATION 1. Ivor-Lewis esophagogastrectomy. 2. Feeding jejunostomy. 3. Placement of two right-s...|
|      2| PREOPERATIVE DIAGNOSES: 1. Gastroesophageal reflux disease. 2. Chronic dyspepsia. POSTOPERATIVE ...|
|      3| PROCEDURE: Colonoscopy. PREOPERATIVE DIAGNOSES: Rectal bleeding and perirectal abscess. POSTOPER...|
|      4| PREOPERATIVE DIAGNOSIS: Right colon tumor. POSTOPERATIVE DIAGNOSES: 1. Right colon cancer. 2. As...|
+-------+-----------------------------------------------------------------

In [11]:
# Run NER pipeline
print("Running NER pipeline... This may take several minutes...")
result_df = ner_pipeline.fit_transform(spark_df)

print("✅ NER pipeline completed")
result_df.select("text_id", "text", "chunk_clinical", "chunk_deid", "chunk_posology").show(5, truncate=100)

Running NER pipeline... This may take several minutes...
✅ NER pipeline completed


25/11/09 20:33:44 WARN DAGScheduler: Broadcasting large task binary with size 1070.4 KiB
25/11/09 20:33:44 WARN TaskSetManager: Stage 18 contains a task of very large size (2002 KiB). The maximum recommended task size is 1000 KiB.


+-------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+----------+----------------------------------------------------------------------------------------------------+
|text_id|                                                                                                text|                                                                                      chunk_clinical|chunk_deid|                                                                                      chunk_posology|
+-------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+----------+----------------------------------------------------------------------------------------------------+
|      0| PROCEDURES PERFORM

# Section 4: Entity Extraction & Merging

This section extracts entities from NER results and merges them with priority:
1. **Posology** (Drug, Dosage) - Highest priority
2. **DeID** (PHI entities) - Medium priority
3. **Clinical** (other clinical entities) - Lowest priority

## 4.1 Extract Entities from NER Results

Extract entities from all three models and merge them according to priority rules.

In [12]:
# Extract entities from NER results with priority merging
# Priority: Posology > DeID > Clinical

def extract_entities_from_ner_results(result_df, text_df):
    """
    Extract entities from NER pipeline results and create entity dataframe.
    Priority: Posology > DeID > Clinical
    
    Args:
        result_df: Spark DataFrame with NER results
        text_df: Pandas DataFrame with text data
        
    Returns:
        Pandas DataFrame with merged entities
    """
    entities_list = []
    
    # Collect clinical entities
    clinical_results = result_df.select(
        "text_id",
        F.explode(F.arrays_zip(
            result_df["chunk_clinical"].result,
            result_df["chunk_clinical"].begin,
            result_df["chunk_clinical"].end,
            result_df["chunk_clinical"].metadata
        )).alias("clinical_chunk")
    ).select(
        "text_id",
        F.expr("clinical_chunk['0']").alias("chunk"),
        F.expr("clinical_chunk['1']").alias("begin"),
        F.expr("clinical_chunk['2']").alias("end"),
        F.expr("clinical_chunk['3']['entity']").alias("entity")
    ).collect()
    
    # Process clinical entities
    clinical_entities = {}
    for row in clinical_results:
        text_id = row.text_id
        if text_id not in clinical_entities:
            clinical_entities[text_id] = []
        clinical_entities[text_id].append({
            'begin': row.begin,
            'end': row.end,
            'chunk': row.chunk,
            'entity': row.entity,
            'source': 'clinical'
        })
    
    # Process DeID entities
    deid_results = result_df.select(
        "text_id",
        F.explode(F.arrays_zip(
            result_df["chunk_deid"].result,
            result_df["chunk_deid"].begin,
            result_df["chunk_deid"].end,
            result_df["chunk_deid"].metadata
        )).alias("deid_chunk")
    ).select(
        "text_id",
        F.expr("deid_chunk['0']").alias("chunk"),
        F.expr("deid_chunk['1']").alias("begin"),
        F.expr("deid_chunk['2']").alias("end"),
        F.expr("deid_chunk['3']['entity']").alias("entity")
    ).collect()
    
    deid_entities = {}
    for row in deid_results:
        text_id = row.text_id
        if text_id not in deid_entities:
            deid_entities[text_id] = []
        deid_entities[text_id].append({
            'begin': row.begin,
            'end': row.end,
            'chunk': row.chunk,
            'entity': row.entity,
            'source': 'deid'
        })
    
    # Process Posology entities (Drug, Dosage only)
    posology_results = result_df.select(
        "text_id",
        F.explode(F.arrays_zip(
            result_df["chunk_posology"].result,
            result_df["chunk_posology"].begin,
            result_df["chunk_posology"].end,
            result_df["chunk_posology"].metadata
        )).alias("posology_chunk")
    ).select(
        "text_id",
        F.expr("posology_chunk['0']").alias("chunk"),
        F.expr("posology_chunk['1']").alias("begin"),
        F.expr("posology_chunk['2']").alias("end"),
        F.expr("posology_chunk['3']['entity']").alias("entity")
    ).filter(
        F.col("entity").isin(["Drug", "Dosage"])
    ).collect()
    
    posology_entities = {}
    for row in posology_results:
        text_id = row.text_id
        if text_id not in posology_entities:
            posology_entities[text_id] = []
        posology_entities[text_id].append({
            'begin': row.begin,
            'end': row.end,
            'chunk': row.chunk,
            'entity': row.entity,
            'source': 'posology'
        })
    
    # Merge entities with priority: Posology > DeID > Clinical
    all_entities = []
    for text_id in text_df['text_id'].values:
        merged = {}
        
        # Add posology entities (highest priority)
        if text_id in posology_entities:
            for ent in posology_entities[text_id]:
                key = (ent['begin'], ent['end'])
                merged[key] = ent
        
        # Add DeID entities (medium priority)
        if text_id in deid_entities:
            for ent in deid_entities[text_id]:
                key = (ent['begin'], ent['end'])
                if key not in merged:  # Don't override posology
                    merged[key] = ent
        
        # Add clinical entities (lowest priority)
        if text_id in clinical_entities:
            for ent in clinical_entities[text_id]:
                key = (ent['begin'], ent['end'])
                if key not in merged:  # Don't override posology or deid
                    merged[key] = ent
        
        # Convert to list
        for ent in merged.values():
            all_entities.append({
                'text_id': text_id,
                'begin': ent['begin'],
                'end': ent['end'],
                'chunk': ent['chunk'],
                'entity': ent['entity']
            })
    
    entity_df = pd.DataFrame(all_entities)
    return entity_df


# Extract entities
print("Extracting entities from NER results...")
entity_df = extract_entities_from_ner_results(result_df, text_df)
print(f"✅ Extracted {len(entity_df)} entities")
print(f"\nEntity types: {sorted(entity_df['entity'].unique())}")
print(f"\nEntity distribution:")
print(entity_df['entity'].value_counts())
entity_df.head(10)

Extracting entities from NER results...


25/11/09 20:33:56 WARN TaskSetManager: Stage 19 contains a task of very large size (2002 KiB). The maximum recommended task size is 1000 KiB.
25/11/09 20:39:52 WARN TaskSetManager: Stage 20 contains a task of very large size (2002 KiB). The maximum recommended task size is 1000 KiB.
25/11/09 20:45:32 WARN TaskSetManager: Stage 21 contains a task of very large size (2002 KiB). The maximum recommended task size is 1000 KiB.
ERROR:root:KeyboardInterrupt while sending command.                 (0 + 4) / 4]
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 718, in readinto
 

KeyboardInterrupt: 

# Section 5: CoNLL File Generation

This section converts NER predictions to CoNLL format for model training. The CoNLL format is a standard format for NER training data.

## 5.1 CoNLL Converter Module

This module converts entity annotations to CoNLL format. It:
1. Tags text with entity markers
2. Tokenizes text using Spark NLP
3. Converts to CoNLL format with BIO tagging scheme

In [ ]:
"""
CoNLL Converter Module
Converts NER predictions to CoNLL format for model training
Based on: 1.3.prepare_CoNLL_from_annotations_for_NER.ipynb
"""

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, List, Dict
from collections import Counter
from tqdm import tqdm
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from pyspark.ml import Pipeline


class CoNLLConverter:
    """Convert NER predictions to CoNLL format"""
    
    def __init__(self, spark: SparkSession):
        """
        Initialize CoNLL Converter
        
        Args:
            spark: SparkSession instance
        """
        self.spark = spark
    
    def make_conll(self, 
                   text_df: pd.DataFrame, 
                   entity_df: pd.DataFrame,
                   save_tag: bool = True,
                   save_conll: bool = True,
                   output_path: str = "data/conll/conll2003_text_file.conll",
                   verbose: bool = None,
                   begin_deviation: int = 0,
                   end_deviation: int = 0) -> str:
        """
        Create CoNLL file from text and entity dataframes
        
        Args:
            text_df: DataFrame with columns ['text_id', 'text']
            entity_df: DataFrame with columns ['text_id', 'begin', 'end', 'chunk', 'entity']
            save_tag: Whether to save tagged text CSV
            save_conll: Whether to save CoNLL file
            output_path: Path to save CoNLL file
            verbose: Whether to print verbose output
            begin_deviation: Deviation to add to begin positions
            end_deviation: Deviation to add to end positions
            
        Returns:
            CoNLL formatted string
        """
        # Prepare dataframes
        df_text = text_df.iloc[:, [0, 1]].copy()
        df_entity = entity_df.iloc[:, [0, 1, 2, 3, 4]].copy()
        df_text.columns = ['text_id', 'text']
        df_entity.columns = ['text_id', 'begin', 'end', 'chunk', 'entity']
        entity_list = list(df_entity.entity.unique())
        
        # Step 1: Tag transformation
        print("Text tagging starting. Applying entities to whole text...\n")
        df = self._apply_tag_ner(df_text, df_entity, save=save_tag, 
                                verbose=verbose, begin_deviation=begin_deviation,
                                end_deviation=end_deviation)
        
        # Step 2: Spark Pipeline for tokenization
        print("\n\nSpark pipeline is running...")
        df_final = self._spark_pipeline(df)
        
        # Step 3: Build CoNLL
        print("Conll file is being created...\n")
        conll_text = self._build_conll(df_final, entity_list, save=save_conll, 
                                       output_path=output_path)
        
        return conll_text
    
    def _transform_text(self, text: str, entities: pd.DataFrame, 
                       verbose: Optional[bool] = None,
                       begin_deviation: int = 0,
                       end_deviation: int = 0) -> str:
        """Transform text by adding entity tags"""
        tag_list = []
        
        # Sort entities by end position (descending) to insert from end to beginning
        entities_sorted = entities.sort_values(by='end', ascending=False)
        
        for _, entity in entities_sorted.iterrows():
            begin = int(entity['begin']) + begin_deviation
            end = int(entity['end']) + end_deviation
            chunk = entity['chunk']
            tag = entity['entity']
            
            # Insert end tag
            text = text[:end] + f' </END_NER:{tag}> ' + text[end:]
            # Insert start tag
            text = text[:begin] + f' <START_NER:{tag}> ' + text[begin:]
            tag_list.append(tag)
        
        sum_of_added_entity = Counter(tag_list)
        sum_of_entity = Counter(entities['entity'].values)
        
        if verbose:
            print(f'Processed text id   : {entities.text_id.values[:1]}')
            print(f'Original Entities   : {sum_of_entity}\nAdded Entities      : {sum_of_added_entity}')
            print(f'Number Equality     : {sum_of_added_entity == sum_of_entity}')
            print("==" * 40)
        
        if not sum_of_entity == sum_of_added_entity:
            print("There is a problem in text id:")
            print(entities.text_id.values[0])
            raise Exception("Check this text!")
        
        return text
    
    def _apply_tag_ner(self, df_text: pd.DataFrame, df_entity: pd.DataFrame,
                      save: bool = False, verbose: Optional[bool] = None,
                      begin_deviation: int = 0, end_deviation: int = 0) -> pd.DataFrame:
        """Apply NER tags to text dataframe"""
        for text_id in tqdm(df_text.text_id):
            text = df_text.loc[df_text['text_id'] == text_id]['text'].values[0]
            entities = df_entity.loc[(df_entity['text_id'] == text_id)].sort_values(
                by='begin', ascending=False
            )
            
            df_text.loc[df_text['text_id'] == text_id, 'text'] = self._transform_text(
                text, entities, verbose=verbose, 
                begin_deviation=begin_deviation, 
                end_deviation=end_deviation
            )
        
        if save:
            output_path = Path("data/processed/text_with_ner_tag.csv")
            output_path.parent.mkdir(parents=True, exist_ok=True)
            df_text.to_csv(output_path, index=False, encoding='utf8')
        
        return df_text
    
    def _spark_pipeline(self, df: pd.DataFrame) -> pd.DataFrame:
        """Run Spark NLP pipeline for tokenization"""
        spark_df = self.spark.createDataFrame(df)
        
        document_assembler = DocumentAssembler()\
            .setInputCol("text")\
            .setOutputCol("document")\
            .setCleanupMode("shrink")
        
        sentence_detector = SentenceDetector()\
            .setInputCols(['document'])\
            .setOutputCol('sentences')\
            .setExplodeSentences(True)
        
        tokenizer = Tokenizer()\
            .setInputCols(["sentences"])\
            .setOutputCol("token")
        
        nlp_pipeline = Pipeline(stages=[document_assembler, sentence_detector, tokenizer])
        
        empty_df = self.spark.createDataFrame([['']]).toDF("text")
        pipeline_model = nlp_pipeline.fit(empty_df)
        
        result = pipeline_model.transform(spark_df.select(['text']))
        
        # Convert Spark DataFrame to pandas without using toPandas()
        # Collect results and create pandas DataFrame manually to avoid Py4JError
        collected = result.select('token.result').collect()
        token_results = [row['result'] for row in collected]
        
        # Create pandas DataFrame
        df_final = pd.DataFrame({'result': token_results})
        
        return df_final
    
    def _build_conll(self, df_final: pd.DataFrame, tag_list: List[str],
                    save: bool = False, output_path: str = "data/conll/conll2003_text_file.conll") -> str:
        """Build CoNLL formatted string"""
        header = "-DOCSTART- -X- -X- O\n\n"
        conll_text = ""
        chunks = []
        tag = 'O'  # token tag
        ct = 'B'   # chunk tag part B or I
        
        for sentence_tokens in tqdm(df_final.result[:]):
            for token in sentence_tokens:
                if token.startswith("<START_NER:"):
                    tag = token.split(':')[1][:-1]
                    if tag not in tag_list:
                        tag = 'O'
                        conll_text += f'{token} NN NN {tag}\n'
                    continue
                
                if token.startswith("</END_NER:") and tag != 'O':
                    for i, chunk in enumerate(chunks):
                        ct = 'B' if i == 0 else 'I'
                        conll_text += f'{chunk} NNP NNP {ct}-{tag}\n'
                    chunks = []
                    tag = 'O'
                    continue
                
                if tag != 'O':
                    chunks.append(token)
                    continue
                
                if tag == 'O':
                    conll_text += f'{token} NN NN {tag}\n'
                    continue
            
            conll_text += '\n'
        
        if save:
            output_path = Path(output_path)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            with open(output_path, "w+", encoding='utf8') as f:
                f.write(header)
                f.write(conll_text)
        
        print("\nDONE!")
        return conll_text
    
    def convert_from_ner_results(self, result_df, text_id_col: str = "text_id"):
        """
        Alternative method: Convert directly from NER results DataFrame
        This method matches tokens with entities based on position
        
        Args:
            result_df: Spark DataFrame with NER results
            text_id_col: Name of text ID column
            
        Returns:
            CoNLL formatted string
        """
        # This is an alternative implementation based on the notebook
        # It tokenizes first, then matches tokens with entities
        pass



## 5.2 Generate CoNLL File

Convert the extracted entities to CoNLL format. The CoNLL format uses BIO tagging scheme where:
- `B-ENTITY`: Beginning of an entity
- `I-ENTITY`: Inside an entity
- `O`: Outside any entity

In [ ]:
# Create CoNLL converter
converter = CoNLLConverter(spark)

# Generate CoNLL file
conll_path = "data/conll/conll2003_text_file.conll"
conll_text = converter.make_conll(
    text_df=text_df,
    entity_df=entity_df,
    save_tag=True,
    save_conll=True,
    output_path=conll_path,
    verbose=False
)

print(f"✅ CoNLL file generated: {conll_path}")
print(f"CoNLL text length: {len(conll_text)} characters")
print("\nSample CoNLL format:")
print(conll_text[:500])

## 5.3 Verify CoNLL File

Read and verify the generated CoNLL file using Spark NLP's CoNLL reader.

In [ ]:
from sparknlp.training import CoNLL

# Read CoNLL file
conll_data = CoNLL().readDataset(spark, conll_path)
print(f"✅ CoNLL file loaded: {conll_data.count()} sentences")
conll_data.show(3, truncate=100)

# Section 6: Custom NER Model Training

This section trains a custom NER model using the generated CoNLL file. It includes:
1. Loading clinical word embeddings
2. Creating training pipeline
3. Training the model
4. Evaluating model performance

## 6.1 Model Trainer Module

This module handles training of custom NER models from CoNLL data.

In [ ]:
# Optimize batch size for GPU if available
try:
    import torch
    use_gpu = torch.cuda.is_available()
    if use_gpu:
        # For better generalization and to reduce overfitting, use smaller batch size
        # Smaller batches provide better gradient estimates and reduce false positives
        batch_size = 8  # Reduced from 16 for better generalization
        print("🚀 GPU detected - using optimized batch size for better generalization")
    else:
        batch_size = 4  # Reduced from 8 for better gradient estimates
        print("Using CPU mode - optimized batch size for better learning")
except:
    batch_size = 4
    use_gpu = False

# OPTIMIZED HYPERPARAMETERS based on Error Analysis:
# Error Analysis showed:
# - 470 False Negatives (model too conservative) → Need more training, lower LR
# - 441 False Positives (model too aggressive) → Need better generalization
# - 502 Misclassifications (entity confusion) → Need more careful learning
#
# Optimizations:
# 1. Lower learning rate (0.001) → Slower, more careful learning, better generalization
# 2. Smaller batch size (4-8) → Better gradient estimates, less overfitting
# 3. More epochs (35) → More learning opportunities
# 4. Higher patience (6) → Allow more epochs for improvement
# 5. Stricter early stopping (0.02) → Select better models
# 6. More validation data (0.25) → Better validation metrics

# Create training pipeline with optimized hyperparameters
training_pipeline = trainer.create_training_pipeline(
    max_epochs=35,  # Increased from 20: More learning opportunities
    lr=0.001,  # Reduced from 0.003: Slower learning, better generalization, less overfitting
    batch_size=batch_size,  # Reduced: Better gradient estimates, less overfitting
    random_seed=0,
    verbose=1,
    test_dataset=test_data_path,
    output_logs_path="./ner_logs",
    validation_split=0.25,  # Increased from 0.2: More validation data for better metrics
    use_best_model=True,
    early_stopping_criterion=0.02,  # Reduced from 0.04: Stricter criterion, select better models
    early_stopping_patience=6  # Increased from 3: Allow more epochs for improvement
)

print("✅ Training pipeline created with OPTIMIZED hyperparameters")
print("\n📊 Optimized Training Parameters (based on Error Analysis):")
print(f"  - Max epochs: 35 (↑ from 20) - More learning opportunities")
print(f"  - Learning rate: 0.001 (↓ from 0.003) - Slower, more careful learning")
print(f"  - Batch size: {batch_size} (↓ for better generalization)")
print(f"  - Validation split: 0.25 (↑ from 0.2) - More validation data")
print(f"  - Early stopping criterion: 0.02 (↓ from 0.04) - Stricter model selection")
print(f"  - Early stopping patience: 6 (↑ from 3) - More epochs allowed")
print(f"\n💡 Expected improvements:")
print(f"  - Reduced False Positives (better generalization)")
print(f"  - Reduced False Negatives (more training)")
print(f"  - Reduced Misclassifications (more careful learning)")
if use_gpu:
    print(f"\n🚀 GPU acceleration: Enabled")

## 6.2 Load Embeddings and Prepare Dataset

Load clinical word embeddings and prepare the CoNLL dataset for training.

In [ ]:
# Initialize model trainer
trainer = ModelTrainer(spark)

# Load clinical embeddings
clinical_embeddings = trainer.load_embeddings("embeddings_clinical")

# Load CoNLL dataset
training_data = trainer.load_conll_dataset(conll_path)

# Split dataset into train and test
train_data, test_data = trainer.split_dataset(training_data, train_ratio=0.8, seed=100)

# Save test data as parquet for evaluation
from pathlib import Path
test_data_path = "data/processed/test_data.parquet"
Path(test_data_path).parent.mkdir(parents=True, exist_ok=True)
clinical_embeddings.transform(test_data).write.parquet(test_data_path)
print(f"✅ Test data saved to {test_data_path}")

## 6.3 Train Custom NER Model

Train a custom NER model using the CoNLL data. The training includes:
- Early stopping to prevent overfitting
- Best model selection based on validation performance
- Extended evaluation logs for monitoring

In [ ]:
# Optimize batch size for GPU if available
try:
    import torch
    use_gpu = torch.cuda.is_available()
    if use_gpu:
        # GPU can handle larger batch sizes
        batch_size = 16
        print("🚀 GPU detected - using optimized batch size for GPU acceleration")
    else:
        batch_size = 8
        print("Using CPU mode - standard batch size")
except:
    batch_size = 8
    use_gpu = False

# Create training pipeline
training_pipeline = trainer.create_training_pipeline(
    max_epochs=20,
    lr=0.003,
    batch_size=batch_size,  # Optimized for GPU if available
    random_seed=0,
    verbose=1,
    test_dataset=test_data_path,
    output_logs_path="./ner_logs",
    validation_split=0.1,
    use_best_model=True,
    early_stopping_criterion=0.04,
    early_stopping_patience=3
)

print("✅ Training pipeline created")
print("Training parameters:")
print(f"  - Max epochs: 20")
print(f"  - Learning rate: 0.003")
print(f"  - Batch size: {batch_size} {'(GPU optimized)' if use_gpu else '(CPU)'}")
print(f"  - Validation split: 0.1")
print(f"  - Early stopping: Enabled")
if use_gpu:
    print(f"  - GPU acceleration: Enabled")

In [ ]:
# Train the model
try:
    import torch
    if torch.cuda.is_available():
        print("🚀 Starting model training with GPU acceleration...")
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
        print("   This will be significantly faster than CPU training")
    else:
        print("Starting model training on CPU... This may take several minutes...")
except:
    print("Starting model training... This may take several minutes...")

trained_model = trainer.train_model(train_data, training_pipeline)

print("\n✅ Model training completed!")
print("\nTraining logs saved to ./ner_logs/")

## 6.4 View Training Logs

View the training logs to see the training progress and metrics.

In [ ]:
# View training logs
import glob

log_files = glob.glob("./ner_logs/MedicalNerApproach*")
if log_files:
    latest_log = max(log_files, key=os.path.getctime)
    print(f"Latest training log: {latest_log}\n")
    with open(latest_log, 'r') as f:
        print(f.read())
else:
    print("No training logs found.")

## 6.5 Evaluate Model Performance

Evaluate the trained model on the test set using precision, recall, and F1-score metrics.

In [ ]:
# Evaluate model
eval_results = trainer.evaluate_model(test_data, drop_o=True, case_sensitive=True)

print("\n✅ Model evaluation completed!")

## 6.6 Save Trained Model

Save the trained model for future use.

In [ ]:
# Save model
model_path = "models/trained/custom_ner_model"
trainer.save_model(model_path)

print(f"\n✅ Model saved to {model_path}")

# Section 7: Resume Training

This section demonstrates how to resume training on additional data. This is useful when:
- You want to continue training on new data from the same taxonomy
- You want to fine-tune a model trained on a different dataset

## 7.1 Resume Training on Same Taxonomy

Load a saved model and continue training on additional data from the same taxonomy.

In [ ]:
# Split test data for resume training demonstration
(test_data_1, test_data_2) = test_data.randomSplit([0.5, 0.5], seed=100)

# Save test data as parquet for easy testing
test_data_1_path = "data/processed/test_1.parquet"
test_data_2_path = "data/processed/test_2.parquet"

clinical_embeddings.transform(test_data_1).write.parquet(test_data_1_path)
clinical_embeddings.transform(test_data_2).write.parquet(test_data_2_path)

print(f"✅ Test data split and saved")
print(f"  - test_data_1: {test_data_1.count()} sentences")
print(f"  - test_data_2: {test_data_2.count()} sentences")

In [ ]:
# Create resume training pipeline with pretrained model
resume_trainer = ModelTrainer(spark)
resume_trainer.load_embeddings("embeddings_clinical")

# Create pipeline that loads the previously trained model
resume_pipeline = resume_trainer.create_training_pipeline(
    max_epochs=2,
    lr=0.003,
    batch_size=8,
    random_seed=0,
    verbose=1,
    test_dataset=test_data_2_path,
    output_logs_path="./ner_logs_resume",
    pretrained_model_path=model_path  # Load previously trained model
)

print("✅ Resume training pipeline created")
print(f"  - Pretrained model: {model_path}")
print(f"  - Additional training epochs: 2")

In [ ]:
# Resume training on additional data
print("Resuming training on additional data...")
resume_trained_model = resume_trainer.train_model(test_data_1, resume_pipeline)

print("\n✅ Resume training completed!")

## 7.2 Evaluate Resumed Model

Evaluate the model after resume training to see if performance improved.

In [ ]:
# Evaluate resumed model
resume_eval_results = resume_trainer.evaluate_model(
    test_data_2, drop_o=True, case_sensitive=True
)

print("\n✅ Resume model evaluation completed!")

## 7.3 Train Using Different Pretrained Model

You can also train a model starting from a different pretrained model (e.g., ner_jsl) and fine-tune it on your data.

In [ ]:
# Load a different pretrained model (e.g., ner_jsl)
jsl_ner = MedicalNerModel.pretrained('ner_jsl', 'en', 'clinical/models')
print(f"\n✅ Loaded ner_jsl model")
print(f"Entity classes in ner_jsl: {len(jsl_ner.getClasses())} classes")
print(f"Sample classes: {jsl_ner.getClasses()[:10]}")

# Note: To use this model as a starting point, you would need to:
# 1. Get the model path: jsl_ner.getModelPath()
# 2. Use it in create_training_pipeline with pretrained_model_path
# 3. Set overrideExistingTags=True if entity tags don't align

# Section 8: Bonus - Visualization, Comparison & Error Analysis

This section includes bonus features for advanced analysis:
1. **Visualize Predictions** - Visualize custom NER model predictions on sample texts
2. **Performance Comparison** - Compare model performance before and after fine-tuning
3. **Error Analysis** - Identify common mistakes and misclassifications

## 8.1 Visualization Module

This module provides functions to visualize NER predictions and analyze model performance.


In [ ]:
"""
Visualization and Analysis Module
Provides functions for visualizing NER predictions, comparing models, and error analysis
"""

import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
from sparknlp_jsl.annotator import MedicalNerModel
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from sparknlp_jsl.annotator import NerConverter
from pyspark.sql import functions as F
from pyspark.ml import Pipeline

# Set style for better visualizations
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")


class NERVisualizer:
    """Visualize NER predictions and analyze model performance"""

    def __init__(self, spark):
        """
        Initialize NER Visualizer

        Args:
            spark: SparkSession instance
        """
        self.spark = spark

    def visualize_predictions(
        self,
        model,
        texts: list,
        embeddings_model,
        max_texts: int = 5,
        save_path: str = None,
    ):
        """
        Visualize NER predictions on sample texts using Spark NLP Display

        Args:
            model: Trained MedicalNerModel
            texts: List of text strings to visualize
            embeddings_model: Word embeddings model
            max_texts: Maximum number of texts to visualize
            save_path: Optional path to save visualization
        """
        print(
            f"Visualizing predictions on {min(len(texts), max_texts)} sample texts..."
        )

        # Remove duplicate texts while preserving order
        seen = set()
        unique_texts = []
        for text in texts:
            if text not in seen:
                seen.add(text)
                unique_texts.append(text)
        texts = unique_texts

        # Limit number of texts
        texts = texts[:max_texts]

        # Create Spark DataFrame with index to track original texts
        text_df = pd.DataFrame({"text": texts})
        text_df["text_id"] = range(len(texts))
        spark_df = self.spark.createDataFrame(text_df)

        # Create pipeline for visualization
        document_assembler = (
            DocumentAssembler()
            .setInputCol("text")
            .setOutputCol("document")
            .setCleanupMode("shrink")
        )

        sentence_detector = (
            SentenceDetector()
            .setInputCols(["document"])
            .setOutputCol("sentence")
            .setExplodeSentences(True)
        )

        tokenizer = Tokenizer().setInputCols(["sentence"]).setOutputCol("token")

        ner_converter = (
            NerConverter()
            .setInputCols(["document", "token", "ner"])
            .setOutputCol("entities")
        )

        pipeline = Pipeline(
            stages=[
                document_assembler,
                sentence_detector,
                tokenizer,
                embeddings_model,
                model,
                ner_converter,
            ]
        )

        # Transform data
        result = pipeline.fit(spark_df).transform(spark_df)

        # Group results by original text (text_id) to avoid duplicates
        # Since explodeSentences creates multiple rows per text, we need to group them
        from pyspark.sql import functions as F
        
        # Collect all results and group by original text
        all_results = result.select("text_id", "text", "entities").collect()
        
        # Group by text_id to combine entities from all sentences of the same text
        text_results = {}
        for row in all_results:
            text_id = row.text_id
            text_content = row.text
            entities = row.entities if row.entities else []
            
            if text_id not in text_results:
                text_results[text_id] = {
                    "text": text_content,
                    "entities": []
                }
            
            # Add entities from this row (avoid duplicates)
            if entities:
                for entity in entities:
                    # Check if entity already exists (same text and position)
                    entity_key = (entity.result, entity.begin, entity.end)
                    existing_keys = [(e.result, e.begin, e.end) for e in text_results[text_id]["entities"]]
                    if entity_key not in existing_keys:
                        text_results[text_id]["entities"].append(entity)

        # Display using Spark NLP Display
        try:
            from sparknlp_display import NerVisualizer

            for text_idx, (text_id, data) in enumerate(sorted(text_results.items())):
                print(f"\n{'='*80}")
                print(f"Text {text_idx + 1}:")
                print(f"{'='*80}")
                print(data["text"])
                print(f"\nEntities found:")

                # Extract entities
                entities = data["entities"]
                if entities:
                    for entity in entities:
                        print(
                            f"  - {entity.result}: {entity.metadata.get('entity', 'N/A')} "
                            f"(confidence: {entity.metadata.get('confidence', 'N/A')})"
                        )
                else:
                    print("  No entities found")

                # Try to use NerVisualizer if available
                try:
                    # Create a temporary row-like object for visualization
                    class TempRow:
                        def __init__(self, entities):
                            self.entities = entities
                    
                    visualizer = NerVisualizer()
                    temp_row = TempRow(entities)
                    visualizer.display(
                        temp_row,
                        label_col="entities",
                        document_col="text",
                    )
                except:
                    print("  (Visual display not available)")
        except ImportError:
            print("Spark NLP Display not available. Showing text output only.")
            for text_idx, (text_id, data) in enumerate(sorted(text_results.items())):
                print(f"\n{'='*80}")
                print(f"Text {text_idx + 1}:")
                print(f"{'='*80}")
                print(data["text"])
                if data["entities"]:
                    for entity in data["entities"]:
                        print(
                            f"  - {entity.result}: {entity.metadata.get('entity', 'N/A')}"
                        )

    def compare_models(
        self,
        model1,
        model1_name: str,
        model2,
        model2_name: str,
        test_data,
        embeddings_model,
    ):
        """
        Compare performance of two models

        Args:
            model1: First model (e.g., pretrained)
            model1_name: Name of first model
            model2: Second model (e.g., fine-tuned)
            model2_name: Name of second model
            test_data: Test dataset
            embeddings_model: Word embeddings model
        """
        from sparknlp_jsl.eval import NerDLMetrics

        print(f"Comparing {model1_name} vs {model2_name}...")

        results = {}

        # Evaluate model 1
        print(f"\nEvaluating {model1_name}...")
        pipeline1 = Pipeline(stages=[embeddings_model, model1])
        pred1 = pipeline1.fit(test_data).transform(
            embeddings_model.transform(test_data)
        )

        evaler = NerDLMetrics(mode="full_chunk")
        eval1 = evaler.computeMetricsFromDF(
            pred1.select("label", "ner"),
            prediction_col="ner",
            label_col="label",
            drop_o=True,
            case_sensitive=True,
        )

        results[model1_name] = eval1

        # Evaluate model 2
        print(f"\nEvaluating {model2_name}...")
        pipeline2 = Pipeline(stages=[embeddings_model, model2])
        pred2 = pipeline2.fit(test_data).transform(
            embeddings_model.transform(test_data)
        )

        eval2 = evaler.computeMetricsFromDF(
            pred2.select("label", "ner"),
            prediction_col="ner",
            label_col="label",
            drop_o=True,
            case_sensitive=True,
        )

        results[model2_name] = eval2

        # Create comparison DataFrame
        comparison_data = []
        for model_name, eval_df in results.items():
            for row in eval_df.collect():
                comparison_data.append(
                    {
                        "Model": model_name,
                        "Entity": row.entity,
                        "Precision": row.precision,
                        "Recall": row.recall,
                        "F1": row.f1,
                        "TP": row.tp,
                        "FP": row.fp,
                        "FN": row.fn,
                    }
                )

        comparison_df = pd.DataFrame(comparison_data)

        # Create visualization
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(
            f"Model Comparison: {model1_name} vs {model2_name}",
            fontsize=16,
            fontweight="bold",
        )

        # 1. F1 Score Comparison
        ax1 = axes[0, 0]
        pivot_f1 = comparison_df.pivot(index="Entity", columns="Model", values="F1")
        pivot_f1.plot(kind="bar", ax=ax1, width=0.8)
        ax1.set_title("F1 Score Comparison by Entity", fontweight="bold")
        ax1.set_ylabel("F1 Score")
        ax1.set_xlabel("Entity Type")
        ax1.legend(title="Model")
        ax1.grid(axis="y", alpha=0.3)
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha="right")

        # 2. Precision Comparison
        ax2 = axes[0, 1]
        pivot_prec = comparison_df.pivot(
            index="Entity", columns="Model", values="Precision"
        )
        pivot_prec.plot(kind="bar", ax=ax2, width=0.8, color=["#3498db", "#e74c3c"])
        ax2.set_title("Precision Comparison by Entity", fontweight="bold")
        ax2.set_ylabel("Precision")
        ax2.set_xlabel("Entity Type")
        ax2.legend(title="Model")
        ax2.grid(axis="y", alpha=0.3)
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha="right")

        # 3. Recall Comparison
        ax3 = axes[1, 0]
        pivot_rec = comparison_df.pivot(
            index="Entity", columns="Model", values="Recall"
        )
        pivot_rec.plot(kind="bar", ax=ax3, width=0.8, color=["#2ecc71", "#f39c12"])
        ax3.set_title("Recall Comparison by Entity", fontweight="bold")
        ax3.set_ylabel("Recall")
        ax3.set_xlabel("Entity Type")
        ax3.legend(title="Model")
        ax3.grid(axis="y", alpha=0.3)
        plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha="right")

        # 4. Overall Metrics
        ax4 = axes[1, 1]
        macro_metrics = []
        for model_name in [model1_name, model2_name]:
            model_data = comparison_df[comparison_df["Model"] == model_name]
            macro_metrics.append(
                {
                    "Model": model_name,
                    "Macro F1": model_data["F1"].mean(),
                    "Macro Precision": model_data["Precision"].mean(),
                    "Macro Recall": model_data["Recall"].mean(),
                }
            )

        macro_df = pd.DataFrame(macro_metrics)
        macro_df.set_index("Model").plot(kind="bar", ax=ax4, width=0.8)
        ax4.set_title("Overall Macro-Average Metrics", fontweight="bold")
        ax4.set_ylabel("Score")
        ax4.set_xlabel("Model")
        ax4.legend(title="Metric")
        ax4.grid(axis="y", alpha=0.3)
        plt.setp(ax4.xaxis.get_majorticklabels(), rotation=0)

        plt.tight_layout()
        plt.show()

        # Print summary
        print(f"\n{'='*80}")
        print("COMPARISON SUMMARY")
        print(f"{'='*80}")
        print(f"\n{model1_name}:")
        print(
            f"  Macro F1: {comparison_df[comparison_df['Model']==model1_name]['F1'].mean():.4f}"
        )
        print(
            f"  Macro Precision: {comparison_df[comparison_df['Model']==model1_name]['Precision'].mean():.4f}"
        )
        print(
            f"  Macro Recall: {comparison_df[comparison_df['Model']==model1_name]['Recall'].mean():.4f}"
        )

        print(f"\n{model2_name}:")
        print(
            f"  Macro F1: {comparison_df[comparison_df['Model']==model2_name]['F1'].mean():.4f}"
        )
        print(
            f"  Macro Precision: {comparison_df[comparison_df['Model']==model2_name]['Precision'].mean():.4f}"
        )
        print(
            f"  Macro Recall: {comparison_df[comparison_df['Model']==model2_name]['Recall'].mean():.4f}"
        )

        return comparison_df

    def error_analysis(self, model, test_data, embeddings_model, top_errors: int = 10):
        """
        Perform error analysis to identify common mistakes

        Args:
            model: Trained model
            test_data: Test dataset with labels
            embeddings_model: Word embeddings model
            top_errors: Number of top errors to display
        """
        from sparknlp_jsl.eval import NerDLMetrics
        from sparknlp_jsl.annotator import NerConverter
        from sparknlp.base import DocumentAssembler
        from sparknlp.annotator import SentenceDetector, Tokenizer

        print("Performing error analysis...")

        # Check if test_data already has document, sentence, token columns
        # If not, we need to create them
        test_columns = test_data.columns

        if (
            "document" in test_columns
            and "sentence" in test_columns
            and "token" in test_columns
        ):
            # Data already tokenized, just add embeddings and model
            print("  Using existing tokenized data...")
            # Convert NER predictions to chunks
            ner_converter_pred = (
                NerConverter()
                .setInputCols(["document", "token", "ner"])
                .setOutputCol("ner_chunks")
            )

            # Convert labels to chunks
            ner_converter_label = (
                NerConverter()
                .setInputCols(["document", "token", "label"])
                .setOutputCol("label_chunks")
            )

            # Create pipeline with existing tokenization
            pipeline = Pipeline(
                stages=[
                    embeddings_model,
                    model,
                    ner_converter_pred,
                    ner_converter_label,
                ]
            )

            # Transform data
            predictions = pipeline.fit(test_data).transform(
                embeddings_model.transform(test_data)
            )
        else:
            # Need to tokenize from scratch
            print("  Tokenizing data...")
            document_assembler = (
                DocumentAssembler()
                .setInputCol("text")
                .setOutputCol("document")
                .setCleanupMode("shrink")
            )

            sentence_detector = (
                SentenceDetector()
                .setInputCols(["document"])
                .setOutputCol("sentence")
                .setExplodeSentences(True)
            )

            tokenizer = Tokenizer().setInputCols(["sentence"]).setOutputCol("token")

            # Convert NER predictions to chunks
            ner_converter_pred = (
                NerConverter()
                .setInputCols(["document", "token", "ner"])
                .setOutputCol("ner_chunks")
            )

            # Convert labels to chunks
            ner_converter_label = (
                NerConverter()
                .setInputCols(["document", "token", "label"])
                .setOutputCol("label_chunks")
            )

            # Create pipeline
            pipeline = Pipeline(
                stages=[
                    document_assembler,
                    sentence_detector,
                    tokenizer,
                    embeddings_model,
                    model,
                    ner_converter_pred,
                    ner_converter_label,
                ]
            )

            # Transform data
            predictions = pipeline.fit(test_data).transform(test_data)

        # Extract predictions and labels from chunks
        errors = {
            "false_positives": defaultdict(list),  # Predicted but not in label
            "false_negatives": defaultdict(list),  # In label but not predicted
            "misclassifications": defaultdict(list),  # Wrong entity type
        }

        # Collect all predictions and labels
        # Use chunks if available, otherwise use token-level
        try:
            # Try to get chunks first (preferred method)
            pred_label_pairs = predictions.select(
                "ner_chunks", "label_chunks", "ner", "label", "token", "document"
            ).collect()
            use_chunks = True
            print("  Using chunk-based analysis...")
        except Exception as e:
            # Fallback to token-level analysis
            print(f"  Chunks not available, using token-level analysis...")
            pred_label_pairs = predictions.select(
                "ner", "label", "token", "document"
            ).collect()
            use_chunks = False

        for row in pred_label_pairs:
            labels = {}
            predictions_dict = {}

            if use_chunks and hasattr(row, "label_chunks") and row.label_chunks:
                # Extract label chunks with entity types
                if row.label_chunks:
                    for chunk in row.label_chunks:
                        # Get entity type from metadata
                        entity_type = chunk.metadata.get("entity", "O")
                        # Handle B-I-O format: extract entity name from "B-ENTITY" or "I-ENTITY"
                        if entity_type and entity_type != "O":
                            # Remove B- or I- prefix if present
                            if entity_type.startswith("B-"):
                                entity_type = entity_type[2:]
                            elif entity_type.startswith("I-"):
                                entity_type = entity_type[2:]

                            if entity_type != "O":  # Skip O labels
                                # Use begin and end for matching, chunk text as value
                                key = (chunk.begin, chunk.end)
                                labels[key] = {
                                    "text": chunk.result,
                                    "entity": entity_type,
                                }

                # Extract prediction chunks with entity types
                if row.ner_chunks:
                    for chunk in row.ner_chunks:
                        # Get entity type from metadata
                        entity_type = chunk.metadata.get("entity", "O")
                        # Handle B-I-O format: extract entity name from "B-ENTITY" or "I-ENTITY"
                        if entity_type and entity_type != "O":
                            # Remove B- or I- prefix if present
                            if entity_type.startswith("B-"):
                                entity_type = entity_type[2:]
                            elif entity_type.startswith("I-"):
                                entity_type = entity_type[2:]

                            if entity_type != "O":  # Skip O labels
                                key = (chunk.begin, chunk.end)
                                predictions_dict[key] = {
                                    "text": chunk.result,
                                    "entity": entity_type,
                                }
            else:
                # Fallback: Extract from token-level labels (B-I-O format)
                # Helper function to extract entities from token-level B-I-O labels
                def extract_entities_from_tokens(label_tokens, tokens):
                    """Extract entities from token-level B-I-O labels"""
                    entities = {}
                    if not label_tokens or not tokens:
                        return entities

                    current_entity = None
                    current_tokens = []
                    current_start = None

                    for i, label_token in enumerate(label_tokens):
                        if i >= len(tokens):
                            break

                        token = tokens[i]
                        # Get label string
                        if hasattr(label_token, "result"):
                            label_str = label_token.result
                        else:
                            label_str = str(label_token)

                        # Get token text and position
                        if hasattr(token, "result"):
                            token_text = token.result
                            token_begin = (
                                token.begin if hasattr(token, "begin") else None
                            )
                            token_end = token.end if hasattr(token, "end") else None
                        else:
                            token_text = str(token)
                            token_begin = None
                            token_end = None

                        if label_str.startswith("B-"):
                            # Save previous entity if exists
                            if (
                                current_entity
                                and current_tokens
                                and current_start is not None
                            ):
                                # Use position-based key
                                if token_begin is not None:
                                    key = (current_start, token_begin - 1)
                                else:
                                    key = (
                                        len(" ".join(current_tokens)),
                                        len(" ".join(current_tokens))
                                        + len(" ".join(current_tokens)),
                                    )
                                entities[key] = {
                                    "text": " ".join(current_tokens),
                                    "entity": current_entity,
                                }

                            # Start new entity
                            current_entity = label_str[2:]  # Remove 'B-' prefix
                            current_tokens = [token_text]
                            current_start = token_begin

                        elif label_str.startswith("I-") and current_entity:
                            # Continue current entity
                            entity_name = label_str[2:]  # Remove 'I-' prefix
                            if entity_name == current_entity:
                                current_tokens.append(token_text)

                        elif label_str == "O" or label_str == "":
                            # Save previous entity if exists
                            if (
                                current_entity
                                and current_tokens
                                and current_start is not None
                            ):
                                if token_begin is not None:
                                    key = (current_start, token_begin - 1)
                                else:
                                    key = (
                                        len(" ".join(current_tokens)),
                                        len(" ".join(current_tokens))
                                        + len(" ".join(current_tokens)),
                                    )
                                entities[key] = {
                                    "text": " ".join(current_tokens),
                                    "entity": current_entity,
                                }
                            current_entity = None
                            current_tokens = []
                            current_start = None

                    # Save last entity if exists
                    if current_entity and current_tokens and current_start is not None:
                        if hasattr(tokens[-1], "end") and tokens[-1].end is not None:
                            key = (current_start, tokens[-1].end)
                        else:
                            key = (
                                current_start,
                                current_start + len(" ".join(current_tokens)),
                            )
                        entities[key] = {
                            "text": " ".join(current_tokens),
                            "entity": current_entity,
                        }

                    return entities

                # Extract labels
                if (
                    hasattr(row, "label")
                    and row.label
                    and hasattr(row, "token")
                    and row.token
                ):
                    labels = extract_entities_from_tokens(row.label, row.token)

                # Extract predictions
                if (
                    hasattr(row, "ner")
                    and row.ner
                    and hasattr(row, "token")
                    and row.token
                ):
                    predictions_dict = extract_entities_from_tokens(row.ner, row.token)

            # Find false positives (predicted but not in label)
            for key, pred_info in predictions_dict.items():
                if key not in labels:
                    errors["false_positives"][pred_info["entity"]].append(
                        pred_info["text"]
                    )

            # Find false negatives (in label but not predicted)
            for key, label_info in labels.items():
                if key not in predictions_dict:
                    errors["false_negatives"][label_info["entity"]].append(
                        label_info["text"]
                    )

            # Find misclassifications (same position, different entity type)
            for key in set(labels.keys()) & set(predictions_dict.keys()):
                label_entity = labels[key]["entity"]
                pred_entity = predictions_dict[key]["entity"]
                if label_entity != pred_entity:
                    errors["misclassifications"][
                        f"{label_entity} → {pred_entity}"
                    ].append(labels[key]["text"])

        # Visualize errors
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle("Error Analysis: Common Mistakes", fontsize=16, fontweight="bold")

        # False Positives
        ax1 = axes[0]
        fp_counts = {k: len(v) for k, v in errors["false_positives"].items()}
        if fp_counts:
            fp_df = pd.DataFrame(list(fp_counts.items()), columns=["Entity", "Count"])
            fp_df = fp_df.sort_values("Count", ascending=False).head(top_errors)
            fp_df.plot(x="Entity", y="Count", kind="barh", ax=ax1, color="#e74c3c")
            ax1.set_title(
                "False Positives (Predicted but not in label)", fontweight="bold"
            )
            ax1.set_xlabel("Count")
            ax1.grid(axis="x", alpha=0.3)

        # False Negatives
        ax2 = axes[1]
        fn_counts = {k: len(v) for k, v in errors["false_negatives"].items()}
        if fn_counts:
            fn_df = pd.DataFrame(list(fn_counts.items()), columns=["Entity", "Count"])
            fn_df = fn_df.sort_values("Count", ascending=False).head(top_errors)
            fn_df.plot(x="Entity", y="Count", kind="barh", ax=ax2, color="#f39c12")
            ax2.set_title(
                "False Negatives (In label but not predicted)", fontweight="bold"
            )
            ax2.set_xlabel("Count")
            ax2.grid(axis="x", alpha=0.3)

        # Misclassifications
        ax3 = axes[2]
        mis_counts = {k: len(v) for k, v in errors["misclassifications"].items()}
        if mis_counts:
            mis_df = pd.DataFrame(
                list(mis_counts.items()), columns=["Error Type", "Count"]
            )
            mis_df = mis_df.sort_values("Count", ascending=False).head(top_errors)
            mis_df.plot(x="Error Type", y="Count", kind="barh", ax=ax3, color="#9b59b6")
            ax3.set_title("Misclassifications (Wrong entity type)", fontweight="bold")
            ax3.set_xlabel("Count")
            ax3.grid(axis="x", alpha=0.3)
            plt.setp(ax3.yaxis.get_majorticklabels(), fontsize=8)

        plt.tight_layout()
        plt.show()

        # Print detailed error examples
        print(f"\n{'='*80}")
        print("TOP ERROR EXAMPLES")
        print(f"{'='*80}")

        print("\n🔴 False Positives (Top 5):")
        for entity, examples in list(errors["false_positives"].items())[:5]:
            print(f"  {entity}: {Counter(examples).most_common(3)}")

        print("\n🟡 False Negatives (Top 5):")
        for entity, examples in list(errors["false_negatives"].items())[:5]:
            print(f"  {entity}: {Counter(examples).most_common(3)}")

        print("\n🟣 Misclassifications (Top 5):")
        for error_type, examples in list(errors["misclassifications"].items())[:5]:
            print(f"  {error_type}: {Counter(examples).most_common(3)}")

        return errors


print("✅ NERVisualizer module loaded")

## 8.2 Visualize Custom NER Model Predictions

Visualize predictions of the custom trained model on sample texts from the dataset.


In [ ]:
# Initialize visualizer
visualizer = NERVisualizer(spark)

# Get sample texts from the dataset (remove duplicates)
sample_texts = text_df['text'].drop_duplicates().head(5).tolist()

# Load the trained model
from pathlib import Path
trained_model_path = "models/trained/custom_ner_model"
if Path(trained_model_path).exists():
    custom_model = MedicalNerModel.load(trained_model_path)
    print("✅ Loaded custom trained model")
    
    # Visualize predictions
    visualizer.visualize_predictions(
        model=custom_model,
        texts=sample_texts,
        embeddings_model=clinical_embeddings,
        max_texts=5
    )
else:
    print("⚠️  Trained model not found. Please train the model first (Section 6).")


## 8.3 Compare Model Performance: Before vs After Fine-tuning

Compare the performance of a pretrained model with the fine-tuned custom model.


In [ ]:
# Compare pretrained model vs custom trained model
from pathlib import Path

# Load pretrained model (baseline)
pretrained_model = MedicalNerModel.pretrained("ner_clinical", "en", "clinical/models")\
    .setInputCols(["document", "token", "embeddings"])\
    .setOutputCol("ner")

# Load custom trained model
trained_model_path = "models/trained/custom_ner_model"
if Path(trained_model_path).exists():
    custom_model = MedicalNerModel.load(trained_model_path)\
        .setInputCols(["document", "token", "embeddings"])\
        .setOutputCol("ner")
    
    print("Comparing models...")
    print("  - Baseline: ner_clinical (pretrained)")
    print("  - Custom: Fine-tuned model on your data")
    
    # Perform comparison
    comparison_results = visualizer.compare_models(
        model1=pretrained_model,
        model1_name="Pretrained (ner_clinical)",
        model2=custom_model,
        model2_name="Custom Fine-tuned",
        test_data=test_data,
        embeddings_model=clinical_embeddings
    )
    
    print("\n✅ Model comparison completed!")
    print("\nKey Insights:")
    improvement = comparison_results[comparison_results['Model']=='Custom Fine-tuned']['F1'].mean() - \
                  comparison_results[comparison_results['Model']=='Pretrained (ner_clinical)']['F1'].mean()
    if improvement > 0:
        print(f"  ✅ Custom model improved F1 by {improvement:.4f}")
    else:
        print(f"  ⚠️  Custom model F1 decreased by {abs(improvement):.4f}")
        print("     Consider: more training data, hyperparameter tuning, or more epochs")
else:
    print("⚠️  Trained model not found. Please train the model first (Section 6).")


## 8.4 Error Analysis: Identify Common Mistakes

Perform detailed error analysis to identify patterns in misclassifications and improve the model.


In [ ]:
# Perform error analysis on custom trained model
from pathlib import Path

trained_model_path = "models/trained/custom_ner_model"
if Path(trained_model_path).exists():
    custom_model = MedicalNerModel.load(trained_model_path)\
        .setInputCols(["document", "token", "embeddings"])\
        .setOutputCol("ner")
    
    print("Performing error analysis on custom trained model...")
    print("This may take a few minutes...")
    
    # Run error analysis
    error_results = visualizer.error_analysis(
        model=custom_model,
        test_data=test_data,
        embeddings_model=clinical_embeddings,
        top_errors=10
    )
    
    print("\n✅ Error analysis completed!")
    print("\n💡 Recommendations based on error analysis:")
    
    # Analyze errors and provide recommendations
    total_fp = sum(len(v) for v in error_results['false_positives'].values())
    total_fn = sum(len(v) for v in error_results['false_negatives'].values())
    total_mis = sum(len(v) for v in error_results['misclassifications'].values())
    
    print(f"\n  Total Errors:")
    print(f"    - False Positives: {total_fp}")
    print(f"    - False Negatives: {total_fn}")
    print(f"    - Misclassifications: {total_mis}")
    
    # Show top error entities (filter out 'O' if it appears)
    if error_results['false_positives']:
        print(f"\n  🔴 Top False Positive Entities:")
        fp_sorted = sorted(error_results['false_positives'].items(), 
                          key=lambda x: len(x[1]), reverse=True)
        # Filter out 'O' entity type
        fp_sorted = [(e, ex) for e, ex in fp_sorted if e != 'O'][:5]
        for entity, examples in fp_sorted:
            print(f"     - {entity}: {len(examples)} errors")
            if examples:
                unique_examples = list(set(examples))[:3]  # Show unique examples
                print(f"       Examples: {unique_examples}")
    
    if error_results['false_negatives']:
        print(f"\n  🟡 Top False Negative Entities:")
        fn_sorted = sorted(error_results['false_negatives'].items(), 
                          key=lambda x: len(x[1]), reverse=True)
        # Filter out 'O' entity type
        fn_sorted = [(e, ex) for e, ex in fn_sorted if e != 'O'][:5]
        for entity, examples in fn_sorted:
            print(f"     - {entity}: {len(examples)} errors")
            if examples:
                unique_examples = list(set(examples))[:3]  # Show unique examples
                print(f"       Examples: {unique_examples}")
    
    if error_results['misclassifications']:
        print(f"\n  🟣 Top Misclassifications:")
        mis_sorted = sorted(error_results['misclassifications'].items(), 
                           key=lambda x: len(x[1]), reverse=True)[:5]
        for error_type, examples in mis_sorted:
            print(f"     - {error_type}: {len(examples)} errors")
            if examples:
                unique_examples = list(set(examples))[:3]  # Show unique examples
                print(f"       Examples: {unique_examples}")
    else:
        print(f"\n  ✅ No misclassifications found")
    
    # Recommendations
    if total_fp > total_fn:
        print("\n  📌 More False Positives detected:")
        print("     → Model is too aggressive in predictions")
        print("     → Consider: Increase confidence threshold, add more negative examples")
        print("     → Action: Review FP entities above and add negative training examples")
    elif total_fn > total_fp:
        print("\n  📌 More False Negatives detected:")
        print("     → Model is too conservative")
        print("     → Consider: More training data, lower confidence threshold")
        print("     → Action: Review FN entities above and add more training examples")
        print("     → Specific actions:")
        if error_results['false_negatives']:
            fn_entities = sorted(error_results['false_negatives'].items(), 
                               key=lambda x: len(x[1]), reverse=True)[:3]
            for entity, _ in fn_entities:
                if entity != 'O':
                    print(f"        - Add more {entity} examples to training data")
    
    if total_mis > 0:
        print("\n  📌 Misclassifications found:")
        print("     → Model confuses between entity types")
        print("     → Action: Add more training examples for confused entity pairs shown above")
    else:
        print("\n  ✅ No misclassifications found - model correctly identifies entity boundaries")
    
    # Additional insights
    print(f"\n  📊 Error Distribution:")
    if error_results['false_positives']:
        fp_entities = [e for e in error_results['false_positives'].keys() if e != 'O']
        print(f"     - FP affects {len(fp_entities)} entity types")
    if error_results['false_negatives']:
        fn_entities = [e for e in error_results['false_negatives'].keys() if e != 'O']
        print(f"     - FN affects {len(fn_entities)} entity types")
    
else:
    print("⚠️  Trained model not found. Please train the model first (Section 6).")


## 8.5 Additional Visualizations: Training Progress

Visualize training progress from training logs.


In [ ]:
# Visualize training progress from logs
import glob
import re

log_files = glob.glob("./ner_logs/MedicalNerApproach*")
if log_files:
    latest_log = max(log_files, key=os.path.getctime)
    print(f"Analyzing training log: {latest_log}")
    
    # Parse training log
    epochs = []
    train_losses = []
    test_f1_scores = []
    
    with open(latest_log, 'r') as f:
        content = f.read()
        
        # Extract epoch information - try to match epoch and loss together
        # Pattern 1: Epoch X/Y followed by loss on same or next line
        epoch_loss_pattern = r'Epoch (\d+)/(\d+).*?avg training loss: ([\d.]+)'
        epoch_loss_matches = re.findall(epoch_loss_pattern, content, re.DOTALL)
        
        # Pattern 2: F1 scores
        f1_pattern = r'Micro-average.*f1: ([\d.]+)'
        f1_found = re.findall(f1_pattern, content)
        
        # Extract epochs and losses that are paired together
        for epoch, total, loss in epoch_loss_matches:
            epoch_num = int(epoch)
            if epoch_num not in epochs:  # Avoid duplicates
                epochs.append(epoch_num)
                train_losses.append(float(loss))
        
        # Extract F1 scores - match with epochs if possible
        # Try to find epoch numbers near F1 scores
        f1_with_epoch_pattern = r'Epoch (\d+)/(\d+).*?Micro-average.*f1: ([\d.]+)'
        f1_with_epoch = re.findall(f1_with_epoch_pattern, content, re.DOTALL)
        
        # Create a mapping of epoch to F1 score
        epoch_to_f1 = {}
        for epoch, total, f1 in f1_with_epoch:
            epoch_num = int(epoch)
            if epoch_num not in epoch_to_f1:
                epoch_to_f1[epoch_num] = float(f1)
        
        # Match F1 scores with epochs we have
        for epoch in epochs:
            if epoch in epoch_to_f1:
                test_f1_scores.append(epoch_to_f1[epoch])
            elif len(test_f1_scores) < len(f1_found):
                # Fallback: use F1 scores in order if available
                test_f1_scores.append(float(f1_found[len(test_f1_scores)]))
        
        # Ensure all lists have the same length for epochs that have losses
        # Only keep epochs that have corresponding loss values
        if len(epochs) != len(train_losses):
            # This shouldn't happen with the new parsing, but just in case
            min_len = min(len(epochs), len(train_losses))
            epochs = epochs[:min_len]
            train_losses = train_losses[:min_len]
    
    if epochs and train_losses:
        # Ensure all lists have matching lengths
        min_len = min(len(epochs), len(train_losses))
        epochs_plot = epochs[:min_len]
        train_losses_plot = train_losses[:min_len]
        
        # Create visualization
        num_plots = 1 + (1 if test_f1_scores else 0)
        fig, axes = plt.subplots(1, num_plots, figsize=(15, 5))
        if num_plots == 1:
            axes = [axes]
        
        # Training Loss
        ax1 = axes[0]
        ax1.plot(epochs_plot, train_losses_plot, marker='o', linewidth=2, markersize=8, color='#3498db')
        ax1.set_title('Training Loss Over Epochs', fontweight='bold', fontsize=14)
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Average Training Loss')
        ax1.grid(True, alpha=0.3)
        ax1.set_xticks(epochs_plot)
        
        # Test F1 Score
        if test_f1_scores:
            ax2 = axes[1]
            # Match F1 scores with epochs
            f1_epochs = epochs_plot[:len(test_f1_scores)]
            f1_scores_plot = test_f1_scores[:len(f1_epochs)]
            ax2.plot(f1_epochs, f1_scores_plot, marker='s', linewidth=2, markersize=8, color='#2ecc71')
            ax2.set_title('Test F1 Score Over Epochs', fontweight='bold', fontsize=14)
            ax2.set_xlabel('Epoch')
            ax2.set_ylabel('Micro-Average F1 Score')
            ax2.grid(True, alpha=0.3)
            ax2.set_xticks(f1_epochs)
            ax2.set_ylim([0, 1])
        
        plt.tight_layout()
        plt.show()
        
        print("✅ Training progress visualized")
        print(f"\n📊 Parsed {len(epochs_plot)} epochs with training loss data")
        if test_f1_scores:
            best_f1 = max(test_f1_scores)
            best_epoch_idx = test_f1_scores.index(best_f1)
            best_epoch = epochs_plot[best_epoch_idx] if best_epoch_idx < len(epochs_plot) else epochs_plot[0]
            print(f"🏆 Best F1 Score: {best_f1:.4f} at epoch {best_epoch}")
    elif epochs:
        print(f"⚠️  Found {len(epochs)} epochs but no training loss data. Log format may be different.")
    else:
        print("⚠️  Could not parse training log format")
else:
    print("⚠️  No training logs found. Please train the model first (Section 6).")


Exception in thread "RemoteBlock-temp-file-clean-thread" java.lang.OutOfMemoryError: Java heap space
Exception in thread "netty-rpc-env-timeout" 

# Section 9: Summary & Next Steps

## Summary

This notebook demonstrated a complete pipeline for:

1. **Dataset Loading**: Loaded healthcare dataset (mtsamples_classifier)
2. **NER Pipeline**: Executed three pre-trained models (clinical, deid, posology)
3. **Entity Extraction**: Extracted and merged entities with priority
4. **CoNLL Generation**: Converted entities to CoNLL format
5. **Model Training**: Trained a custom NER model from CoNLL data
6. **Resume Training**: Continued training on additional data
7. **Evaluation**: Evaluated model performance with metrics

## Key Features

- **Priority-based Entity Merging**: Posology > DeID > Clinical
- **Early Stopping**: Prevents overfitting during training
- **Best Model Selection**: Automatically selects best model based on validation performance
- **Resume Training**: Continue training on new data
- **Comprehensive Evaluation**: Precision, recall, F1-score metrics

## Next Steps

1. **Fine-tune Hyperparameters**: Adjust learning rate, batch size, epochs
2. **Add More Data**: Include more training data for better performance
3. **Error Analysis**: Analyze misclassifications to improve the model
4. **Visualization**: Visualize predictions on sample texts
5. **Production Deployment**: Deploy the trained model for production use

## Additional Resources

- [Spark NLP Healthcare Documentation](https://nlp.johnsnowlabs.com/docs/en/licensed_install)
- [Healthcare NLP for Data Scientists Course](https://www.udemy.com/course/healthcare-nlp-for-data-scientists/)
- [Spark NLP Workshop](https://github.com/JohnSnowLabs/spark-nlp-workshop)